# 6.2 LDA / SVD / NMF - 주석 설명판

원본 `6_2.ipynb`의 코드에 설명 주석을 단 버전이에요.
6.1의 PCA 말고도 다른 차원 축소 방법 3가지를 붓꽃 데이터로 비교해요.

| 방법 | 축을 찾는 기준 | 정답(레이블) 사용? | 특징 |
|---|---|---|---|
| **PCA** (6.1) | 데이터 전체의 분산이 가장 큰 방향 | ✕ (비지도) | 가장 기본 |
| **LDA** | 클래스끼리는 멀리, 같은 클래스는 가깝게 | **○ (지도)** | 분류용 차원 축소 |
| **SVD** (TruncatedSVD) | 행렬을 특이값 분해해서 큰 특이값만 남김 | ✕ | 평균을 빼지 않음 → 희소 행렬(텍스트)에 많이 씀 |
| **NMF** | 음수가 없는 두 행렬의 곱으로 분해 | ✕ | 입력값이 **0 이상**이어야 함 |

## 1. LDA (Linear Discriminant Analysis, 선형 판별 분석)

**PCA와의 차이**
- PCA: "데이터가 가장 넓게 퍼진 방향" → 클래스는 신경 안 씀
- LDA: "**클래스를 가장 잘 구분하는 방향**" → 정답(target)을 보고 축을 찾음

LDA는 두 가지를 동시에 노려요.
- **클래스 간 분산**(between-class) → 크게 = 품종끼리는 멀리 떨어지게
- **클래스 내부 분산**(within-class) → 작게 = 같은 품종끼리는 뭉치게

⚠ LDA로 만들 수 있는 축의 최대 개수 = **클래스 수 - 1** → 붓꽃은 3품종이라 최대 2개

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris

iris = load_iris()
# PCA처럼 먼저 표준화
iris_scaled = StandardScaler().fit_transform(iris.data)

In [ ]:
lda = LinearDiscriminantAnalysis(n_components=2)

# ⚠ PCA와 달리 fit에 정답(iris.target)도 같이 넣어요 → 지도 학습 방식
lda.fit(iris_scaled, iris.target)
iris_lda = lda.transform(iris_scaled)
print(iris_lda.shape)   # (150, 2) → 피처 4개 → 2개

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# LDA 결과를 DataFrame으로 만들고 품종별로 산점도 (6.1 PCA 그래프와 같은 방식)
lda_columns = ['lda_comp_1', 'lda_comp_2']
irisDF_lda = pd.DataFrame(iris_lda, columns=lda_columns)
irisDF_lda['target'] = iris.target
markers = ['^', 's', 'o']

for i, marker in enumerate(markers):
    x_axis_data = irisDF_lda[irisDF_lda['target'] == i]['lda_comp_1']
    y_axis_data = irisDF_lda[irisDF_lda['target'] == i]['lda_comp_2']
    plt.scatter(x_axis_data, y_axis_data, marker=marker,
                label=iris.target_names[i])

plt.legend()
plt.xlabel('lda_comp_1')
plt.ylabel('lda_comp_2')
plt.show()

# → PCA 그래프보다 세 품종이 가로축(lda_comp_1) 방향으로 더 확실하게 떨어져 있음
#   정답을 보고 '구분이 잘 되는 방향'을 골랐기 때문

## 2. SVD (Singular Value Decomposition, 특이값 분해)

행렬 A를 세 행렬의 곱으로 쪼개는 방법이에요.

$$A = U \Sigma V^T$$

- $\Sigma$(시그마)의 대각선 값 = **특이값** (클수록 중요한 정보)
- **TruncatedSVD**: 특이값이 큰 상위 k개만 남기고 나머지는 잘라냄(truncate) → 차원 축소

**PCA와의 관계**
PCA는 내부적으로 "데이터에서 **평균을 뺀 뒤**(중심화) SVD"를 해요.
TruncatedSVD는 **평균을 빼지 않고** 바로 SVD를 해요.
→ 그래서 희소 행렬(대부분 0인 텍스트 데이터)에 쓰기 좋아요. 평균을 빼면 0이었던 칸이 전부 0이 아닌 값이 되어 버리니까요.

In [ ]:
from sklearn.decomposition import TruncatedSVD, PCA

iris = load_iris()
iris_ftrs = iris.data   # ⚠ 스케일링 안 한 원본 데이터

tsvd = TruncatedSVD(n_components=2)
tsvd.fit(iris_ftrs)
iris_tsvd = tsvd.transform(iris_ftrs)

# c=iris.target: 품종 번호(0,1,2)에 따라 점 색을 다르게 (반복문 없이 한 번에 그리는 방법)
plt.scatter(x=iris_tsvd[:, 0], y=iris_tsvd[:, 1], c=iris.target)
plt.xlabel('TSVD Comp 1')
plt.ylabel('TSVD Comp 2')

# → 원본에 바로 적용해도 품종별로 어느 정도 나뉨

## 3. 스케일링한 데이터로 TruncatedSVD vs PCA 비교

In [ ]:
from sklearn.preprocessing import StandardScaler

# StandardScaler = 평균을 빼고(평균 0) 표준편차로 나눔 → 이미 '중심화'가 된 상태
iris_scaled = StandardScaler().fit_transform(iris_ftrs)

# 같은 데이터에 TruncatedSVD
tsvd = TruncatedSVD(n_components=2)
tsvd.fit(iris_scaled)
iris_tsvd = tsvd.transform(iris_scaled)

# 같은 데이터에 PCA
pca = PCA(n_components=2)
pca.fit(iris_scaled)
iris_pca = pca.transform(iris_scaled)

# 그래프 2개를 가로로 나란히 (ncols=2)
fig, (ax1, ax2) = plt.subplots(figsize=(9, 4), ncols=2)
ax1.scatter(x=iris_tsvd[:, 0], y=iris_tsvd[:, 1], c=iris.target)
ax2.scatter(x=iris_pca[:, 0], y=iris_pca[:, 1], c=iris.target)

ax1.set_title('TSVD Transform')
ax2.set_title('PCA Transform')

# → 두 그래프가 거의 똑같음!
#   이유: 스케일링으로 이미 평균이 0이 됐으니, PCA(평균 빼고 SVD) = TruncatedSVD(그냥 SVD)
#   (축 방향이 뒤집혀 보일 수는 있음 - 부호만 다르고 같은 축)

## 4. NMF (Non-negative Matrix Factorization, 비음수 행렬 분해)

행렬 V를 **음수가 하나도 없는** 두 행렬 W, H의 곱으로 근사해요.

$$V \approx W \times H$$

- W: 각 데이터가 '숨은 특징(성분)'을 얼마나 가지고 있는지 → 이게 축소된 데이터
- H: 각 숨은 특징이 원래 피처들로 어떻게 구성되는지

**⚠ 입력 데이터에 음수가 있으면 안 돼요.**
→ 그래서 아래 코드는 표준화(음수가 생김)한 `iris_scaled`가 아니라 원본 `iris_ftrs`를 써요.

음수가 없어서 "부분들을 더해서 전체를 만든다"는 해석이 쉬워요.
(예: 얼굴 이미지 = 눈 성분 + 코 성분 + 입 성분, 문서 = 주제1 + 주제2 ...) → 이미지 분석, 토픽 모델링, 추천 시스템에 많이 써요.

In [ ]:
from sklearn.decomposition import NMF

nmf = NMF(n_components=2)
nmf.fit(iris_ftrs)                   # 원본 데이터 (모든 값이 0 이상)
iris_nmf = nmf.transform(iris_ftrs)

plt.scatter(x=iris_nmf[:, 0], y=iris_nmf[:, 1], c=iris.target)
plt.xlabel('NMF Comp 1')
plt.ylabel('NMF Comp 2')

# → NMF로도 품종이 꽤 잘 나뉨. 축 값이 전부 0 이상인 것도 확인할 수 있어요.
# (FutureWarning이 뜨는 건 init 기본값이 바뀐다는 안내라 무시해도 됨
#  경고를 없애려면 NMF(n_components=2, init='nndsvda') 처럼 직접 지정)

## 5. 정리

| 방법 | 핵심 | 언제 쓰나 |
|---|---|---|
| **PCA** | 분산 최대 방향 | 일반적인 차원 축소, 시각화 |
| **LDA** | 클래스 구분 최대 방향 (정답 필요) | 분류 문제의 전처리 |
| **TruncatedSVD** | 중심화 없이 SVD | 희소 행렬 (텍스트 TF-IDF 등) |
| **NMF** | 음수 없는 행렬 곱으로 분해 | 이미지, 토픽 모델링, 추천 |

**⚠ 핵심 포인트**
- 스케일링한 데이터에서는 **TruncatedSVD ≈ PCA** (평균이 이미 0이라서)
- LDA만 **fit에 정답(y)을 넣는 지도 방식**이고, 축 개수는 최대 '클래스 수 - 1'
- NMF는 **음수 입력 불가** → 표준화 대신 원본(또는 MinMaxScaler) 사용